# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 1

**Student Name:** AYEN GEOFFREY ALEXANDER  
**Registration Number:** 2024/A/KCS/5102/G/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This notebook covers **Stages 1 to 5** of the Data Pre-processing & Exploratory Data Analysis Pipeline as given in the official companion guide.

We follow the exact order the lecturer set:  
1. Identify the Machine Learning problem and title  
2. Source and describe the dataset  
3. Set up the working environment  
4. Import the required libraries  
5. Load the data and take a first look  

All explanations are written in simple English so every group member can follow.

---
## Stage 1: Identify Your Machine Learning Problem & Title

Every good project starts with a clear question.  
We are not just looking at numbers. We want to answer a real question that someone in Uganda might care about.

**Our problem title:**  
**Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets Using Historical Price and Market Data**

### Why this title is strong
- It names the **target** (the price of maize and the price of beans).
- It names the **task type** (regression – we are predicting continuous numbers).
- It names the **predictors** (historical prices, month, year, market location).
- It is linked to a real-world need (farmers, traders and organisations that buy food need to know future prices).

This follows the guide’s advice: a good problem statement must clearly state the target, the predictors and the task type.

---
## Stage 2: Source and Describe Your Dataset

### Where the data comes from
We use the **Uganda Real-Time Food Prices (RTFP)** dataset.  
It is published by the World Bank Development Economics Data Group.  
The prices are collected from the World Food Programme (WFP) and the Food and Agriculture Organization (FAO).  
Some missing prices are filled by machine-learning estimates, which is why we also have the completed columns that start with `c_`.

Source link (public):  
https://microdata.worldbank.org/catalog/8241  
Also available on the Humanitarian Data Exchange (HDX).

### Why this dataset is suitable
- It is real-world data, not a toy dataset made only for teaching.
- It has more than 500 rows and more than 5 columns.
- It contains genuine data-quality issues (many missing values in the original price columns).
- It relates directly to the machine-learning problem we chose.

These four points match the non-negotiable selection requirements in the companion guide (Stage 2.2).

### Markets we selected
To keep the work focused and high quality we keep only seven markets:
1. Market Average (national picture)
2. Gulu
3. Lira
4. Jinja
5. Hoima
6. Busia
7. Mbarara

These markets give good geographic coverage (north, east, west and national average) and have complete monthly records from 2007 to 2026.

### Target variables
- `c_maize` – completed monthly maize price (UGX per kg)
- `c_beans` – completed monthly beans price (UGX per kg)

We also keep the original observed columns (`maize` and `beans`) so that later notebooks can show proper missing-value analysis.

---
## Stage 3: Set Up Your Working Environment

We can use either Google Colab or a local Jupyter Notebook.  
Both produce the same `.ipynb` file type.  
For this group project we recommend Google Colab because it is free, needs no installation, and makes sharing easy.

**Notebook file name (important for marks):**  
`2024AKCS5102GF_BCS3101_Assignment2_Notebook1.ipynb`

The guide says the name must follow the pattern  
`RegistrationNumber_BCS3101_Assignment2.ipynb`.  
Using the correct name helps us avoid losing the 10% Submission Compliance marks.

---
## Stage 4: Import the Required Libraries

We import every library we will need in the first code cell.  
This is good practice and lets anyone who opens the notebook see the tools at a glance.

The libraries below are the ones listed in the companion guide (Stage 4).

In [ ]:
# Import the libraries we need for this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# These will be used more in later notebooks, but we import them now so the whole pipeline is ready
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Optional but useful for missing-data pictures later
import missingno as msno
from scipy import stats

# Make plots look clean
%matplotlib inline
sns.set_style('whitegrid')

print("All libraries imported successfully.")

**What each library does (simple explanation)**  
- `pandas` – loads and works with tables of data  
- `numpy` – fast number calculations  
- `matplotlib` and `seaborn` – draw charts  
- `scikit-learn` – tools for scaling, encoding, splitting and later modelling  
- `missingno` – special pictures that show where data is missing  
- `scipy.stats` – extra statistical tests if we need them

---
## Stage 5: Load the Data and Take a First Look

Before we clean anything we must first understand what we have.  
The guide says: “This inspection stage is where you form your hypotheses about the data’s quality; you are not yet fixing anything, you are diagnosing.”

In [ ]:
# Load the original Excel file
# Make sure the file UGA_RTFP_mkt_2007_2026-08-24.xlsx is in the same folder as this notebook
# or update the path below if you put the file somewhere else.

df_full = pd.read_excel('UGA_RTFP_mkt_2007_2026-08-24.xlsx')

print("Full dataset loaded.")
print(f"Rows: {df_full.shape[0]}, Columns: {df_full.shape[1]}")

The full file has more than 10 000 rows because it contains many markets.  
We now keep only the seven markets we chose.

In [ ]:
# Keep only the seven markets we decided on
selected_markets = [
    'Market Average',
    'Gulu',
    'Lira',
    'Jinja',
    'Hoima',
    'Busia',
    'Mbarara'
]

df = df_full[df_full['mkt_name'].isin(selected_markets)].copy()

# Reset the index so it is clean
df = df.reset_index(drop=True)

print("After selecting the seven markets:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nMarkets in our working data:")
print(df['mkt_name'].value_counts())

We now have a smaller, focused table.  
Each of the seven markets has 236 monthly records (from 2007 to 2026).  
Total rows = 7 × 236 = 1 652.  
This is still well above the 500-row requirement in the guide.

### First look at the data
The guide lists several commands that every student must run and interpret.  
We run them one by one and write what we see.

In [ ]:
# Look at the first few rows
print("First 5 rows:")
df.head()

The first rows show the structure.  
We can already see columns for market name, year, month, latitude, longitude, and many price columns.

In [ ]:
# Exact shape
print("Shape of our working dataset:")
print(df.shape)
print(f"\nNumber of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Column names and data types
print("Column names and data types:")
df.info()

`df.info()` tells us two important things:  
1. The data type of every column (int, float, object).  
2. How many non-null values each column has.  

We can already see that the original price columns (`beans`, `maize`, etc.) have many missing values, while the completed columns (`c_beans`, `c_maize`, etc.) are full.

In [ ]:
# Summary statistics for numeric columns
print("Summary statistics for numeric columns:")
df.describe()

`df.describe()` gives us the count, mean, standard deviation, minimum, 25 %, 50 % (median), 75 % and maximum for every numeric column.  
This is the first place we can spot strange values (for example a negative price or an extremely high price that looks like a typing error).

In [ ]:
# Exact data types
print("Data types of each column:")
print(df.dtypes)

In [ ]:
# List of all column names
print("All column names:")
print(df.columns.tolist())

Looking at the column names helps us spot any strange characters or inconsistent naming that could cause problems later.

In [ ]:
# Count of missing values in every column
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_pct.round(2)
}).sort_values('missing_percent', ascending=False)

print("Missing values (highest first):")
print(missing_df[missing_df['missing_count'] > 0])

This table is very important.  
It shows exactly how many values are missing and the percentage.  
Later notebooks (especially Notebook 2 on Data Cleaning) will use these numbers to decide whether to fill the missing values or drop the column.

In [ ]:
# Check for exact duplicate rows
dup_count = df.duplicated().sum()
print(f"Number of exact duplicate rows: {dup_count}")

Duplicate rows can make some patterns look stronger than they really are.  
If the count is greater than zero we will remove them in the cleaning notebook.

In [ ]:
# Quick look at our two main target columns
print("Summary of the two main target variables:")
print(df[['c_maize', 'c_beans']].describe())

Both target columns (`c_maize` and `c_beans`) have no missing values.  
Their means, medians and ranges look reasonable for Ugandan market prices in UGX.

In [ ]:
# Save the filtered dataset so the other group members can use the same clean starting point
df.to_csv('uganda_maize_beans_selected_markets.csv', index=False)
print("Filtered dataset saved as 'uganda_maize_beans_selected_markets.csv'")
print("All later notebooks should load this file instead of the original Excel.")

---
## End of Notebook 1

### What we have done in this notebook
- Chose a clear regression problem and wrote a strong title.
- Described the real-world source of the data and why it meets the assignment rules.
- Selected seven markets that give good coverage and enough rows.
- Set up the environment and imported the libraries listed in the guide.
- Loaded the data, filtered it, and ran every first-look command the guide requires.
- Saved a clean starting CSV for the rest of the group.

### What comes next
Notebook 2 (KUKUNDAKWE SAVIOUS) will take the CSV we just saved and perform full **Data Cleaning** (missing values, duplicates and outliers) following Stage 6 of the companion guide.

**Important reminder from the guide**  
Every number we write in the final report must match the output of this notebook exactly.  
Always re-run the notebook from top to bottom before submitting.